# Hybrid Recommender — Content-Based + Collaborative Filtering
**Dataset:** Amazon Electronics (SQLite)

Pipeline:
1. Carregar dados
2. Gerar ratings sintéticos (se não existir tabela de ratings)
3. Content-Based Filtering (CBF)
4. Collaborative Filtering (CF) — item-item
5. Hybrid Recommender (weighted)

## 1. Importações e ligação à base de dados

In [ ]:
import sqlite3, pandas as pd
conn = sqlite3.connect("../../amazon_electronics.db")
print(pd.read_sql("SELECT * FROM ratings LIMIT 3", conn))
conn.close()

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

DB_PATH = "../../amazon_electronics.db"  # ajusta se necessário

conn = sqlite3.connect(DB_PATH)

# Ver todas as tabelas disponíveis
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tabelas na base de dados:", tables["name"].tolist())

df = pd.read_sql_query("SELECT * FROM products", conn)
conn.close()

print(f"\nProdutos carregados: {len(df)}")
df.head()

OperationalError: unable to open database file

## 2. Gerar ratings sintéticos

O dataset de produtos não tem ratings individuais por utilizador.
Geramos ratings sintéticos realistas usando o `rating_count` como proxy de popularidade:
- Produtos mais populares têm maior probabilidade de serem avaliados
- Os ratings são amostrados com distribuição que favorece valores altos (como em dados reais)

In [ ]:
def generate_synthetic_ratings(df, n_users=200, seed=42):
    """
    Gera ratings sintéticos com base na popularidade dos produtos (rating_count).
    Produtos com mais avaliações têm maior probabilidade de serem selecionados.
    """
    np.random.seed(seed)

    product_ids = df["product_id"].values

    # Normalizar rating_count para usar como probabilidade de seleção
    counts = df["rating_count"].fillna(1).values.astype(float)
    probs = counts / counts.sum()

    rows = []
    for user_id in range(1, n_users + 1):
        # Cada utilizador avalia entre 3 e 15 produtos
        n_ratings = np.random.randint(3, 16)

        # Selecionar produtos com probabilidade proporcional à popularidade
        rated_products = np.random.choice(
            product_ids,
            size=min(n_ratings, len(product_ids)),
            replace=False,
            p=probs
        )

        for product_id in rated_products:
            # Rating entre 1 e 5, com distribuição realista (maioria positiva)
            rating = np.random.choice(
                [1, 2, 3, 4, 5],
                p=[0.05, 0.10, 0.20, 0.35, 0.30]
            )
            rows.append({"user_id": user_id, "product_id": product_id, "rating": rating})

    return pd.DataFrame(rows)


# Verificar se já existe uma tabela de ratings na DB
conn = sqlite3.connect(DB_PATH)
existing_tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'", conn
)["name"].tolist()

if "ratings" in existing_tables:
    print("Tabela 'ratings' encontrada na DB — a usar dados reais.")
    ratings = pd.read_sql_query("SELECT * FROM ratings", conn)
else:
    print("Tabela 'ratings' não encontrada — a gerar ratings sintéticos.")
    ratings = generate_synthetic_ratings(df, n_users=200)

conn.close()

print(f"\nTotal de ratings: {len(ratings)}")
print(f"Utilizadores únicos: {ratings['user_id'].nunique()}")
print(f"Produtos avaliados: {ratings['product_id'].nunique()}")
ratings.head()

## 3. Content-Based Filtering (CBF)
Usa categoria e preço para encontrar produtos similares.

In [ ]:
# Preparar features de conteúdo
df_cbf = df.copy()
df_cbf["category"] = df_cbf["category"].str.split(r"[|&]")
df_cbf["category"] = df_cbf["category"].apply(lambda x: list(set(x)))

mlb = MultiLabelBinarizer()
category_matrix = mlb.fit_transform(df_cbf["category"])

scaler = MinMaxScaler()
price_scaled = scaler.fit_transform(df_cbf[["discounted_price"]])

feature_matrix = np.hstack((category_matrix, price_scaled))
content_similarity = cosine_similarity(feature_matrix)

print(f"Feature matrix shape: {feature_matrix.shape}")
print(f"Content similarity matrix: {content_similarity.shape}")

In [ ]:
# Mapa product_id -> índice (partilhado por CBF e CF)
product_index_map = pd.Series(df.index, index=df["product_id"]).to_dict()


def get_cbf_scores(product_id, k=10):
    """
    Retorna dict {product_id: content_similarity_score} para os k produtos mais similares.
    """
    if product_id not in product_index_map:
        return {}

    idx = product_index_map[product_id]
    scores = list(enumerate(content_similarity[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:k + 1]

    return {df.iloc[i]["product_id"]: float(score) for i, score in scores}


# Teste
example_pid = df["product_id"].iloc[0]
print(f"CBF para produto '{example_pid}':")
get_cbf_scores(example_pid, k=5)

## 4. Collaborative Filtering (CF) — Item-Item

Cria a user-item matrix e calcula similaridade entre produtos com base nos padrões de avaliação.

In [ ]:
# 4.1 Construir a user-item matrix
# Apenas produtos que existem em df (garantir consistência)
valid_products = set(df["product_id"])
ratings_clean = ratings[ratings["product_id"].isin(valid_products)].copy()

user_item_matrix = ratings_clean.pivot_table(
    index="user_id",
    columns="product_id",
    values="rating"
)

print(f"User-item matrix: {user_item_matrix.shape} (utilizadores x produtos)")
print(f"Sparsidade: {user_item_matrix.isna().sum().sum() / user_item_matrix.size:.1%} valores em falta")
user_item_matrix.head()

In [ ]:
# 4.2 Preencher NaN com 0 e calcular similaridade item-item
user_item_filled = user_item_matrix.fillna(0)

# .T porque queremos similaridade entre produtos (colunas), não utilizadores (linhas)
cf_similarity_matrix = cosine_similarity(user_item_filled.T)

cf_similarity_df = pd.DataFrame(
    cf_similarity_matrix,
    index=user_item_filled.columns,
    columns=user_item_filled.columns
)

print(f"CF similarity matrix: {cf_similarity_df.shape}")
cf_similarity_df.head()

In [ ]:
def get_cf_scores(rated_products, k=10):
    """
    Dado um dict {product_id: rating}, retorna scores CF para produtos candidatos.
    rated_products: dict {product_id: rating}
    """
    scores = {}

    for product_id, rating in rated_products.items():
        if product_id not in cf_similarity_df.columns:
            continue

        similar = cf_similarity_df[product_id].drop(index=product_id, errors="ignore")
        top_similar = similar.nlargest(k)

        for similar_pid, sim_score in top_similar.items():
            if similar_pid in rated_products:
                continue
            scores[similar_pid] = scores.get(similar_pid, 0) + float(sim_score) * rating

    return scores


# Teste
test_ratings = {df["product_id"].iloc[0]: 4.5, df["product_id"].iloc[5]: 3.0}
cf_result = get_cf_scores(test_ratings, k=10)
print(f"CF produziu {len(cf_result)} candidatos")
sorted(cf_result.items(), key=lambda x: x[1], reverse=True)[:5]

## 5. Hybrid Recommender (Weighted)

Combina CBF e CF com pesos ajustáveis:
```
final_score = weight_cf * cf_score + weight_cbf * cbf_score
```

In [ ]:
def normalize_scores(scores_dict):
    """Normaliza scores para [0, 1]."""
    if not scores_dict:
        return {}
    max_val = max(scores_dict.values())
    if max_val == 0:
        return scores_dict
    return {k: v / max_val for k, v in scores_dict.items()}


def hybrid_recommender(rated_products, n=5, weight_cf=0.6, weight_cbf=0.4):
    """
    Recomendador híbrido: combina Collaborative Filtering e Content-Based Filtering.

    Parâmetros:
        rated_products : dict {product_id: rating} — produtos já avaliados pelo utilizador
        n              : número de recomendações a devolver
        weight_cf      : peso do CF no score final (default 0.6)
        weight_cbf     : peso do CBF no score final (default 0.4)
    """
    k = n * 3  # candidatos intermédios

    # --- CF scores ---
    raw_cf = get_cf_scores(rated_products, k=k)
    norm_cf = normalize_scores(raw_cf)

    # --- CBF scores (agregar sobre todos os produtos avaliados) ---
    raw_cbf = {}
    for product_id, rating in rated_products.items():
        for pid, score in get_cbf_scores(product_id, k=k).items():
            if pid in rated_products:
                continue
            raw_cbf[pid] = raw_cbf.get(pid, 0) + score * rating
    norm_cbf = normalize_scores(raw_cbf)

    # --- Combinar todos os candidatos ---
    all_candidates = set(norm_cf.keys()) | set(norm_cbf.keys())

    final_scores = {}
    for pid in all_candidates:
        cf_score  = norm_cf.get(pid, 0)
        cbf_score = norm_cbf.get(pid, 0)
        final_scores[pid] = weight_cf * cf_score + weight_cbf * cbf_score

    # --- Top N ---
    top_n_ids = sorted(final_scores, key=lambda x: final_scores[x], reverse=True)[:n]

    result = df[df["product_id"].isin(top_n_ids)][[
        "product_id", "product_name", "category", "discounted_price"
    ]].copy()

    result["hybrid_score"]  = result["product_id"].map(final_scores)
    result["cf_score"]      = result["product_id"].map(lambda x: norm_cf.get(x, 0))
    result["cbf_score"]     = result["product_id"].map(lambda x: norm_cbf.get(x, 0))

    return result.sort_values("hybrid_score", ascending=False).reset_index(drop=True)

## 6. Testar o Hybrid Recommender

In [ ]:
# Produtos avaliados pelo utilizador (product_id: rating)
rated_products = {
    "B0789LZTCJ": 4.2,
    "B094JNXNPV": 3.5
}

recommendations = hybrid_recommender(
    rated_products=rated_products,
    n=5,
    weight_cf=0.6,
    weight_cbf=0.4
)

recommendations

In [ ]:
# Comparar: só CBF vs só CF vs Hybrid
print("=" * 60)
print("COMPARAÇÃO: CBF puro vs CF puro vs Hybrid")
print("=" * 60)

print("\n📦 Só Content-Based (weight_cf=0, weight_cbf=1):")
print(hybrid_recommender(rated_products, n=5, weight_cf=0.0, weight_cbf=1.0)[["product_name", "cbf_score"]])

print("\n👥 Só Collaborative Filtering (weight_cf=1, weight_cbf=0):")
print(hybrid_recommender(rated_products, n=5, weight_cf=1.0, weight_cbf=0.0)[["product_name", "cf_score"]])

print("\n⚡ Hybrid (weight_cf=0.6, weight_cbf=0.4):")
print(hybrid_recommender(rated_products, n=5, weight_cf=0.6, weight_cbf=0.4)[["product_name", "hybrid_score"]])